In [ ]:
import pandas
from sklearn.preprocessing import StandardScaler, scale
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

df: pandas.DataFrame = pandas.read_pickle('/mnt/hit4/hit4user/PycharmProjects/mysql_connector/c')
X = df.drop(['system'], 1)
X = X.drop(['under'], 1)
Y = df['system']
print("X ?????, ????????", X.shape)
print("Y", Y.shape)
p = len(Y[Y == 1])
n = len(Y[Y == 0])
print("Y OK", p)
print("Y FAIL", n)
print("FAIL/OK", n // p)
print("Nan exist?", df.isnull().values.any())

kfold = StratifiedKFold(n_splits=5)

# ??????? ??????(???)

In [ ]:
# SCALE
scaler = StandardScaler().fit(X, Y)
X_s = scaler.transform(X)

# CROSS-VAL
# 1 Linear Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter= 340)

results = cross_val_score(model, X_s, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

# ?????? ???????? ??????? ? ?????????????? ???????? ??????????(???)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
params = {'criterion':['gini','entropy'], 'max_leaf_nodes': list(range(4, 20)),
          'min_samples_split': [2, 3, 4], 'max_depth': list(range(3, 14))}
clf = GridSearchCV(DecisionTreeClassifier(), params, cv=kfold)
results = clf.fit(X,Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

# ????????? ???

## 1. ???

In [ ]:
from sklearn.ensemble import RandomForestClassifier
params = {'n_estimators': [5, 50], 'min_samples_split': [2, 3, 4],
          'max_leaf_nodes': list(range(10, 20)), 'max_depth': list(range(3, 13))}
clf = GridSearchCV(RandomForestClassifier(), params, cv=kfold)
results = clf.fit(X_s, Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X_s, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

## 2. ??????????

In [ ]:
from sklearn.ensemble import RandomForestClassifier
X = df.drop(['system'], 1)
X = X.drop(['under'], 1)
Y = df['under']
params = {'n_estimators': [5, 50], 'min_samples_split': [2, 3, 4],
          'max_leaf_nodes': list(range(10, 20)), 'max_depth': list(range(3, 13))}
clf = GridSearchCV(RandomForestClassifier(), params, cv=kfold)
results = clf.fit(X_s, Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X_s, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

# K ????????? ???????(???)

In [ ]:
# SCALE
X = df.drop(['system'], 1)
X = X.drop(['under'], 1)
Y = df['system']
X_s = scale(X)

from sklearn.neighbors import KNeighborsClassifier
params = {'n_neighbors':list(range(6, 11)), 'leaf_size':list(range(5, 40, 5))}
clf = GridSearchCV(KNeighborsClassifier(), params, cv=kfold)
results = clf.fit(X_s, Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X_s, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

# XGBoost ?????? ?????? ?????????? (???)

In [ ]:
import xgboost as xgb
import numpy as np

dtrain = xgb.DMatrix(X, Y)
param = {'booster': 'gbtree', 'objective': 'binary:logistic', 'scale_pos_weight': 1.6, 'max_depth': 3, 'eta': 0.1,
         'gamma': 1}
num_round = 9
res = xgb.cv(param, dtrain, num_round, metrics=['error', 'auc'], nfold=5)
print("cross-train accuracy train\t", 1 - np.mean(res['train-error-mean']))
print("cross-train gini\t", np.mean(res['train-auc-mean'] * 2 - 1))
print("cross-test accuracy test\t", 1 - np.mean(res['test-error-mean']))
print("cross-test gini\t\t", np.mean(res['test-auc-mean'] * 2 - 1))

??? ????? ??????, ??? ?????? ?????????? ??????? ????????? ?? ????????? ? XGBoost

# XGBoost ?????????????? ?????? ??????????

## 1. ??????? ???

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
X = df.drop(['system'], 1)
X = X.drop(['under'], 1)
Y = df['system']
params = {'objective': ['binary:logistic'], 'n_estimators': list(range(3, 20)), 'scale_pos_weight': [32],
          'learning_rate': list(np.arange(0, 1, 0.1)), 'max_depth': list(range(2, 21)), 'gamma': [0, 0.5, 1, 1.2, 1.5, 2]}
clf = RandomizedSearchCV(XGBClassifier(), params, cv=kfold, n_iter=50) # pre_dispatch = 1,
results = clf.fit(X, Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X, Y, cv=kfold)
print("Accuracy: %f" % results.mean())

## 2. ??????? ???????????:

In [ ]:
Y = df['under']
params = {'objective': ['binary:logistic'], 'n_estimators': list(range(3, 20)), 'scale_pos_weight': [45],
          'learning_rate': list(np.arange(0, 1, 0.1)), 'max_depth': list(range(2, 21)), 'gamma': [0, 0.5, 1, 1.2, 1.5, 2]}
kfold = StratifiedKFold(n_splits=5)
clf = RandomizedSearchCV(XGBClassifier(), params, cv=kfold, n_iter=30)
results = clf.fit(X, Y)
print(results.best_estimator_)
results = cross_val_score(results.best_estimator_, X, Y, cv=kfold)
print("Accuracy: %f" % results.mean())